In [18]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F
import torch.nn as nn
import pandas as pd
import torch

import sys
sys.path.append('/data/vision/beery/scratch/antoine/CBM_reid/methods')
from seek_code import SEEK
from backbone import Backbone
from concept_head import ConceptHead
from retrieval import Retrieval
from projector import Projector
from data_handler import EleHandler
from concept_head_tunneled import ConceptHeadTunneled, categorical_CE_loss, CrossedHeadNN, MultiHeadNN



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:

dictonary_path=Path('/data/vision/beery/scratch/antoine/CBM_reid/data_processing/data/out_apr2/image_dictonary.csv')
dictonary = pd.read_csv(dictonary_path)
dictonary


,subject_id,image,ele_id,#season,#capture,anonymized_capture_id,subject-SEEK,ele-SEEK,picture_time,preprocessed_image_path,left_ear_path,right_ear_path
0,92126812,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,1,a7f15c6f92cd6028dc590ecf8dfc386476faa3b4,B00T_1E____-0000X_0S00,B00T11E8080-0000X_0S00,2015-05-22 21:31:22,/data/vision/beery/scratch/antoine/CBM_reid/pr...,NaN,NaN
1,92126813,EFA_IDU/T1000/T1000_R1/EFA_IDU_T1000_R1_IMAG00...,T1000,EFA_IDU,2,4e2838a5c2c0294128f8b72b996c2edfcf498b04,B00T11E8080-0000X00S00,B00T11E8080-0000X_0S00,2015-05-22 21:33:46,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...
2,92126815,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,2,9229c652be571b6970dd2ba7fbe104597dcc851e,B00T11E0000-0000X_0S00,B00T11E____-5000X_0S00,2015-05-22 21:32:50,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...,NaN
3,92126816,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,3,d29ac1e2a0c9da2f40c3f2e1dae60aff100aa042,B00T11E____-0040X_0S00,B00T11E____-5000X_0S00,2015-05-22 21:32:51,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...,NaN
4,92126817,EFA_IDU/T1001/T1001_R1/EFA_IDU_T1001_R1_IMAG00...,T1001,EFA_IDU,4,f73d21ecc60aaf4e940a02283aaeb3ffb4604cee,C00T1_E____-4000X_0S0_,B00T11E____-5000X_0S00,2015-05-22 21:32:52,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
8439,92306374,EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...,F1000,EFA_IDI,8,f31a41505f66afe04009f5382c254d915f6858f8,B00T11E0000-0000X00S00,B00T11E0000-0000X0_S00,2013-08-17 16:46:01,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...
8440,92306375,EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...,F1000,EFA_IDI,9,eda8c2c3d7b5f94ca83e77374ed76058ca05ae06,B00T11E0000-3000X00S00,B00T11E0000-0000X0_S00,2013-08-17 16:46:04,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...,/data/vision/beery/scratch/antoine/CBM_reid/pr...
8441,92306418,EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...,F1000,EFA_IDI,10,9b8001c9c0ba19469b383c0f5e08f1bc0ff70638,B00T11E0000-____X0_S00,B00T11E0000-0000X0_S00,2013-08-17 16:46:35,/data/vision/beery/scratch/antoine/CBM_reid/pr...,NaN,/data/vision/beery/scratch/antoine/CBM_reid/pr...
8442,92306424,EFA_IDI/F1000/F1000_R1/EFA_IDI_F1000_R1_IMAG00...,F1000,EFA_IDI,11,383b7a07da8ba0c646b712e4abdb7a90ac2976dc,B00T1_E0000-____X0_S00,B00T11E0000-0000X0_S00,2013-08-17 16:47:24,/data/vision/beery/scratch/antoine/CBM_reid/pr...,NaN,/data/vision/beery/scratch/antoine/CBM_reid/pr...


# First test : We are predicting with the concept head while correcting to obtain all subject_seeks, then aggregating them (using the function we defined in SEEK), then comparing them with the true ele_seek

In [35]:
code = 'trash'
CH = ConceptHeadTunneled(loss=categorical_CE_loss, print_every=1, experiment_code = code, layer = CrossedHeadNN(), reset_weights=False)
ba = Backbone(model_name="MegaDescriptor", pretraining="backbone_for_concepts_w", experiment_code=code)

dataset = EleHandler(subset=None)
loader = DataLoader(dataset, batch_size=512, shuffle=False)

loading weights for the concept head


/data/vision/beery/scratch/antoine/CBM_reid/methods/backbone.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_path, map_location=self.devi

In [37]:
def perfect_correction(hard_predicted_concepts, subject_SEEK, elephant_SEEK):
    return subject_SEEK

def oracle_correction(hard_predicted_concepts, subject_SEEK, elephant_SEEK):
    return elephant_SEEK

perfect_subject_seek, ele_ids_1 = CH.collect_embeddings(loader, ba, intervention_fn=perfect_correction, aggregate_seeks=False)
_, aggregated_subject_seek, ele_ids_2 = SEEK.aggregate_seek(perfect_subject_seek, ele_ids_1)
elephant_seek, ele_ids_3 = CH.collect_embeddings(loader, ba, intervention_fn=oracle_correction, aggregate_seeks=False)

import pandas as pd
df = pd.DataFrame({
    'perfect_subject_seek': [SEEK(element).__str__() for element in perfect_subject_seek.cpu()],
    'aggregated_subject_seek': [SEEK(element).__str__() for element in aggregated_subject_seek.cpu()],
    'elephant_seek': [SEEK(element).__str__() for element in elephant_seek.cpu()],
    'ele_id_1': ele_ids_1.cpu(),
    'ele_id_2': ele_ids_2,
    'ele_id_3': ele_ids_3.cpu()
})



df.head(50)


collecting concepts from concept head


100%|██████████| 17/17 [01:28<00:00,  5.18s/it]


collecting concepts from concept head


100%|██████████| 17/17 [01:26<00:00,  5.08s/it]


,perfect_subject_seek,aggregated_subject_seek,elephant_seek,ele_id_1,ele_id_2,ele_id_3
0,B00T_1E____-0000X_0S00,B00T11E8080-0000X00S00,B00T11E8080-0000X_0S00,999.0,999.0,999.0
1,B00T11E8080-0000X00S00,B00T11E8080-0000X00S00,B00T11E8080-0000X_0S00,999.0,999.0,999.0
2,B00T11E0000-0000X_0S00,B00T11E0000-4040X_0S00,B00T11E____-5000X_0S00,1000.0,1000.0,1000.0
3,B00T11E____-0040X_0S00,B00T11E0000-4040X_0S00,B00T11E____-5000X_0S00,1000.0,1000.0,1000.0
4,C00T1_E____-4000X_0S0_,B00T11E0000-4040X_0S00,B00T11E____-5000X_0S00,1000.0,1000.0,1000.0
5,B00T11E0000-0000X00S_0,B00T11E8070-4000X00S00,B00T11E7000-0000X_0S00,1001.0,1001.0,1001.0
6,B00T11E8070-4000X_0S00,B00T11E8070-4000X00S00,B00T11E7000-0000X_0S00,1001.0,1001.0,1001.0
7,B00T11E____-4540X_0S00,B00T11E____-4550X_0S00,B00T11E____-4550X_0S00,1002.0,1002.0,1002.0
8,B00T11E____-5050X_0S00,B00T11E____-4550X_0S00,B00T11E____-4550X_0S00,1002.0,1002.0,1002.0
9,B00T11E____-4550X_0S00,B00T11E____-4550X_0S00,B00T11E____-4550X_0S00,1002.0,1002.0,1002.0


In [38]:
CH.accuracy_fn(aggregated_subject_seek.cpu(), elephant_seek.cpu())


{'sex': 0.9970393180847168,
 'age': 0.9860255718231201,
 'right_tusk': 0.9245617985725403,
 'left_tusk': 0.9127190709114075,
 'R_tear_1': 0.7576977610588074,
 'R_hole_1': 0.8404784202575684,
 'R_tear_2': 0.7666982412338257,
 'R_hole_2': 0.9202984571456909,
 'L_tear_1': 0.7886072993278503,
 'L_hole_1': 0.8429654240608215,
 'L_tear_2': 0.7598294615745544,
 'L_hole_2': 0.917693018913269,
 'right_extreme': 0.676811933517456,
 'left_extreme': 0.6775224804878235,
 'ear_special': 0.8758882284164429,
 'body_special': 0.8094505071640015,
 'average': 0.8408929370343685,
 'whole_code': 0.089175745844841,
 'left_ear': 0.7973235368728637,
 'right_ear': 0.7923969626426697}

# Now we will make the same test but directly from the data dictionary. If we get the same results, it shows that it has nothing to do with the concept head

In [ ]:
perfect_subject_seek = torch.stack([SEEK(seek).one_hot_encode() for seek in dictonary['subject-SEEK']])
elephant_seek = torch.stack([SEEK(seek).one_hot_encode() for seek in dictonary['ele-SEEK']])
ele_ids_1 = dictonary['ele_id']


_, aggregated_subject_seek, ele_ids_2 = SEEK.aggregate_seek(perfect_subject_seek, ele_ids_1)

df = pd.DataFrame({
    'perfect_subject_seek': [SEEK(element).__str__() for element in perfect_subject_seek.cpu()],
    'aggregated_subject_seek': [SEEK(element).__str__() for element in aggregated_subject_seek.cpu()],
    'elephant_seek': [SEEK(element).__str__() for element in elephant_seek.cpu()],
    'ele_id_1': ele_ids_1,
    'ele_id_2': ele_ids_2
})
df.head(50)

,perfect_subject_seek,aggregated_subject_seek,elephant_seek,ele_id_1,ele_id_2
0,B00T_1E____-0000X_0S00,B00T_1E____-0000X_0S00,B00T11E8080-0000X_0S00,T1000,T1000
1,B00T11E8080-0000X00S00,B00T11E8080-0000X00S00,B00T11E8080-0000X_0S00,T1000,T1000
2,B00T11E0000-0000X_0S00,B00T11E0000-0000X_0S00,B00T11E____-5000X_0S00,T1001,T1001
3,B00T11E____-0040X_0S00,B00T11E____-0040X_0S00,B00T11E____-5000X_0S00,T1001,T1001
4,C00T1_E____-4000X_0S0_,C00T1_E____-4000X_0S0_,B00T11E____-5000X_0S00,T1001,T1001
5,B00T11E0000-0000X00S_0,B00T11E0000-0000X00S_0,B00T11E7000-0000X_0S00,T1002,T1002
6,B00T11E8070-4000X_0S00,B00T11E8070-4000X_0S00,B00T11E7000-0000X_0S00,T1002,T1002
7,B00T11E____-4540X_0S00,B00T11E____-4540X_0S00,B00T11E____-4550X_0S00,T1003,T1003
8,B00T11E____-5050X_0S00,B00T11E____-5050X_0S00,B00T11E____-4550X_0S00,T1003,T1003
9,B00T11E____-4550X_0S00,B00T11E____-4550X_0S00,B00T11E____-4550X_0S00,T1003,T1003


In [40]:
CH.accuracy_fn(aggregated_subject_seek.cpu(), elephant_seek.cpu())

{'sex': 0.9430364966392517,
 'age': 0.9458787441253662,
 'right_tusk': 0.8498342037200928,
 'left_tusk': 0.8414258360862732,
 'R_tear_1': 0.5495026111602783,
 'R_hole_1': 0.6629559397697449,
 'R_tear_2': 0.5847939252853394,
 'R_hole_2': 0.7310516238212585,
 'L_tear_1': 0.5499762892723083,
 'L_hole_1': 0.6744433641433716,
 'L_tear_2': 0.6004263162612915,
 'L_hole_2': 0.7389862537384033,
 'right_extreme': 0.7257224321365356,
 'left_extreme': 0.7235907316207886,
 'ear_special': 0.8225959539413452,
 'body_special': 0.8010421395301819,
 'average': 0.7340789288282394,
 'whole_code': 0.05637138709425926,
 'left_ear': 0.6574845910072327,
 'right_ear': 0.6508053064346313}

In [ ]:
def aggregate_seek(SEEK_codes, ele_ids):
    def make_seek_df(SEEK_codes, ele_ids):
        data = []
        seeks = []

        for i in range(SEEK_codes.shape[0]):
            row = SEEK(SEEK_codes[i]).categorical_tensor()[0].tolist()
            data.append(row)
            seeks.append(SEEK(SEEK_codes[i]).__str__())
        import pandas as pd
        df = pd.DataFrame(data, columns=SEEK.attribute_names)

        df['subj_seek_str'] = seeks
        if isinstance(ele_ids, torch.Tensor):
            df['ele_id'] = ele_ids.cpu()
        else: 
            df['ele_id'] = ele_ids
            
        df['encounter_id'] = 1

        return df

    # Collect SEEK codes
    df = make_seek_df(SEEK_codes, ele_ids)

    # Copy DataFrame for mapped values
    df_mapped = df.copy()

    # Get mapping dictionary
    mapping = SEEK.mappings

    # Apply mapping correctly (avoiding index shift)
    for col in df.columns:
        if col in mapping:  # Ensure column exists in mapping
            max_idx = len(mapping[col]) - 1
            df_mapped[col] = df[col].apply(lambda x: mapping[col][x] if 0 <= x <= max_idx else None)


    ### HEURISTIC RULES (copied from the original code)
    def get_val(p_counts, u, most_likely, other, cutoff=.8, fraction=1):
        if u in p_counts and p_counts[u] > cutoff:
            return u
        if other in p_counts and most_likely in p_counts:
            if p_counts[other] > (p_counts[most_likely] * fraction):
                return other
            else:
                return most_likely
        if other in p_counts:
            return other
        if most_likely in p_counts:
            return most_likely
        return None

    def get_sex_val(p): return get_val(p, u='_', most_likely='B', other='C', fraction=2)
    def get_age_val(p): return get_val(p, u='_', most_likely='00', other='20')
    def get_tusk_val(p): return get_val(p, u='_', most_likely='1', other='0')

    def get_tear_hole_val(p, cutoff=.9, frac=2):
        u, z = '_', '0'
        if u in p and p[u] > cutoff: return u
        other_sum = sum(value for key, value in p.items() if key != z and key != u)
        if z in p and p[z] > (other_sum * frac): return z
        return max((k for k in p if k not in {u, z}), key=lambda k: p[k], default=None)

    def get_feature_val(p): return get_val(p, u='1', most_likely='0', other='_', fraction=1.5)

    ### FUNCTION TO GENERATE SEEK CODE STRING
    def generate_seek_code(row):
        """Generates a SEEK code string from DataFrame attributes for each row."""
        return (
            f"{row['sex']}{row['age']}T{row['right_tusk']}{row['left_tusk']}"
            f"E{row['R_tear_1']}{row['R_hole_1']}{row['R_tear_2']}{row['R_hole_2']}-"
            f"{row['L_tear_1']}{row['L_hole_1']}{row['L_tear_2']}{row['L_hole_2']}X"
            f"{row['right_extreme']}{row['left_extreme']}S{row['ear_special']}{row['body_special']}"
        )

    ### FUNCTION TO AGGREGATE SEEK CODES AT ELEPHANT LEVEL
    def aggregate_elephant_seek_codes(df):
        """Aggregates subject-level SEEK codes into elephant-level SEEK codes while keeping image order."""
        
        ### FUNCTION TO GENERATE SEEK CODE STRING
    
        elephant_seek_dict = {}

        for ele_id, group in df.groupby('ele_id'):
            if ele_id not in elephant_seek_dict:
                elephant_seek_dict[ele_id] = {}

            for col in df.columns:
                if col not in ['ele_id', 'encounter_id']:
                    counts = group[col].value_counts(normalize=True).to_dict()

                    if col == 'sex':
                        elephant_seek_dict[ele_id][col] = get_sex_val(counts)
                    elif col == 'age':
                        elephant_seek_dict[ele_id][col] = get_age_val(counts)
                    elif col in ['right_tusk', 'left_tusk']:
                        elephant_seek_dict[ele_id][col] = get_tusk_val(counts)
                    elif col in ['R_tear_1', 'R_hole_1', 'L_tear_1', 'L_hole_1']:
                        elephant_seek_dict[ele_id][col] = get_tear_hole_val(counts, cutoff=.9, frac=2)
                    elif col in ['R_tear_2', 'R_hole_2', 'L_tear_2', 'L_hole_2']:
                        elephant_seek_dict[ele_id][col] = get_tear_hole_val(counts, cutoff=.95, frac=2.5)
                    elif col in ['right_extreme', 'left_extreme', 'ear_special', 'body_special']:
                        elephant_seek_dict[ele_id][col] = get_feature_val(counts)
        import pandas as pd
        elephant_seek_df = pd.DataFrame.from_dict(elephant_seek_dict, orient='index')
        elephant_seek_df.index.name = 'ele_id'

        # Merge back the aggregated SEEK codes into the original DataFrame (keeping original order)
        merged_df = df.copy()
        for col in elephant_seek_df.columns:
            merged_df[col] = merged_df['ele_id'].map(elephant_seek_df[col])

        # Generate the SEEK code for each row
        merged_df["SEEK_code"] = merged_df.apply(generate_seek_code, axis=1)


        #print([SEEK(seek).one_hot_encode() for seek in merged_df["SEEK_code"]])
        ele_seek_list = [SEEK(seek).one_hot_encode() for seek in merged_df["SEEK_code"]]
        subject_seek_list = [SEEK(seek).one_hot_encode() for seek in df["subj_seek_str"]]
        ele_id = merged_df["ele_id"].tolist()

        ele_seek = torch.stack(ele_seek_list)
        subject_seek = torch.stack(subject_seek_list)

        return subject_seek, ele_seek, ele_id
    return aggregate_elephant_seek_codes(df_mapped)